# M2 — Agent Layer Demo

End-of-milestone demonstration: run Agent 1 (news sentiment) and Agent 2 (macro regime) and inspect their outputs.

## Prerequisites
```bash
uv run python scripts/init_db.py
uv run python scripts/ingest_prices.py
uv run python scripts/ingest_macro.py
uv run python scripts/ingest_alpha_vantage_news.py  # historical backfill
```

## Sections
1. **Run NewsAgent** — news sentiment signals for one week
2. **Sentiment bar chart** — per-sector scores
3. **Run MacroAgent** — regime classification + rate outlook
4. **Macro regime timeline** — classified regime across the backtest window
5. **Signals table** — all signals written to SQLite

In [ ]:
import datetime
import logging
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..') / 'src'))

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
from IPython.display import display
from sqlalchemy import create_engine, text

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

DB_PATH = Path('..') / 'data' / 'state.db'
engine = create_engine(f'sqlite:///{DB_PATH}')

# Analysis date — must fall within both news_raw and macro date ranges.
ANALYSIS_DATE = datetime.date(2024, 6, 7)

SECTOR_NAMES = {
    'XLK': 'Technology',          'XLF': 'Financials',
    'XLV': 'Health Care',         'XLY': 'Consumer Discretionary',
    'XLP': 'Consumer Staples',    'XLE': 'Energy',
    'XLI': 'Industrials',         'XLB': 'Materials',
    'XLRE': 'Real Estate',        'XLU': 'Utilities',
}

print(f'Analysis date : {ANALYSIS_DATE}')
print(f'DB path       : {DB_PATH.resolve()}')

## Section 1 — Run NewsAgent

Queries `news_raw` for the trailing 7 days, calls `claude-haiku-4-5-20251001` (cached on repeat runs),
validates JSON output, writes 10 signal rows to `signals`.

In [ ]:
from agents.news_agent import NewsAgent

news_agent = NewsAgent()
news_input = news_agent.prepare_input(ANALYSIS_DATE, engine)
coverage = {etf: len(articles) for etf, articles in news_input['sectors'].items()}
total_articles = sum(coverage.values())

print(f'Article coverage for week {news_input["week_start"]} → {news_input["analysis_date"]}:')
for etf, n in sorted(coverage.items()):
    print(f'  {etf:5s}  {n:3d}  {"█" * min(n, 20)}')
print(f'\nTotal: {total_articles} articles across {len(coverage)} sectors')

if total_articles == 0:
    print('\n⚠  No news data — run ingest_alpha_vantage_news.py first.')
    news_result = None
else:
    news_result = news_agent.run(ANALYSIS_DATE, engine)
    print(f'\n✓ NewsAgent done  conviction={news_result["conviction"]:.2f}')
    print(f'  key_themes: {news_result["key_themes"]}')

## Section 2 — Sentiment bar chart

In [ ]:
if news_result is None:
    print('⚠  No result to plot.')
else:
    sentiments = news_result['sector_sentiments']
    labels = [f'{etf} — {SECTOR_NAMES.get(etf, etf)}' for etf in sentiments]
    scores = list(sentiments.values())
    colors = ['#2ecc71' if s > 0.05 else '#e74c3c' if s < -0.05 else '#bdc3c7' for s in scores]

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(labels, scores, color=colors, alpha=0.85, edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlim(-1.05, 1.05)
    ax.set_xlabel('Sentiment  (−1 = strong bearish, +1 = strong bullish)', fontsize=10)
    ax.set_title(
        f'News Sentiment by Sector  |  week ending {ANALYSIS_DATE}  '
        f'|  conviction = {news_result["conviction"]:.2f}',
        fontsize=11,
    )
    for bar, score in zip(bars, scores):
        ha = 'left' if score >= 0 else 'right'
        ax.text(score + (0.02 if score >= 0 else -0.02),
                bar.get_y() + bar.get_height() / 2,
                f'{score:+.2f}', va='center', ha=ha, fontsize=9)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

    print('Key themes:')
    for i, t in enumerate(news_result['key_themes'], 1):
        print(f'  {i}. {t}')

## Section 3 — Run MacroAgent

Pulls 30 days of all 7 FRED series, computes derived features (VIX change, T10Y2Y vs 90d avg,
CPI YoY, etc.), adds an XLF+XLI news digest, then calls `claude-sonnet-4-6`.
The model reasons step-by-step before outputting `regime`, `rate_outlook`, `confidence`, and `rationale`.

In [ ]:
from agents.macro_agent import MacroAgent

macro_agent = MacroAgent()
macro_input = macro_agent.prepare_input(ANALYSIS_DATE, engine)

n_series = sum(len(v) for v in macro_input['series_30d'].values())
n_news   = sum(len(v) for v in macro_input['macro_news_digest'].values())

print(f'Macro data points (30d) : {n_series}')
print(f'Derived features        : {len(macro_input["derived_features"])}')
print(f'Macro news articles     : {n_news}')

if macro_input['derived_features']:
    print('\nKey derived features:')
    for k, v in macro_input['derived_features'].items():
        print(f'  {k:<30s} {v}')

if n_series == 0:
    print('\n⚠  No macro data — run ingest_macro.py first.')
    macro_result = None
else:
    macro_result = macro_agent.run(ANALYSIS_DATE, engine)
    print(f'\n✓ MacroAgent done')
    print(f'  regime      : {macro_result["regime"]}')
    print(f'  rate_outlook: {macro_result["rate_outlook"]}')
    print(f'  confidence  : {macro_result["confidence"]:.2f}')
    print(f'\nRationale:\n  {macro_result["rationale"]}')
    print(f'\nReasoning (excerpt):\n  {macro_result["reasoning"][:400]}...')

## Section 4 — Macro regime timeline

Reads all `macro_regime` signals from the `signals` table and plots a coloured timeline.
Run MacroAgent for multiple dates to populate this chart (see loop at the bottom of this section).

In [ ]:
regime_df = pd.read_sql(
    text("""
        SELECT s.date, s.signal_value, s.confidence,
               r.signal_value as rate_val
        FROM signals s
        LEFT JOIN signals r
            ON r.date = s.date AND r.agent_name = 'macro' AND r.target = 'rate_outlook'
        WHERE s.agent_name = 'macro' AND s.target = 'macro_regime'
        ORDER BY s.date
    """),
    engine,
    parse_dates=['date'],
)

if regime_df.empty:
    print('⚠  No macro regime signals in DB yet.')
    print('   Run MacroAgent.run() for multiple dates to populate this chart.')
    print('   Example loop (run in a script or separate cell):')
    print('     from agents.macro_agent import MacroAgent')
    print('     agent = MacroAgent()')
    print('     for d in pd.date_range("2024-01-05", "2024-06-07", freq="W-FRI"):')
    print('         agent.run(d.date(), engine)')
else:
    REGIME_COLORS = {1.0: '#2ecc71', 0.0: '#f39c12', -1.0: '#e74c3c'}
    REGIME_LABELS = {1.0: 'risk_on', 0.0: 'neutral', -1.0: 'risk_off'}
    RATE_MARKERS  = {1.0: '▲', 0.0: '─', -1.0: '▼'}

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True,
                                    gridspec_kw={'height_ratios': [3, 1]})

    # ── Regime timeline (top) ──────────────────────────────────────────────
    for _, row in regime_df.iterrows():
        color = REGIME_COLORS.get(row['signal_value'], '#95a5a6')
        ax1.axvspan(row['date'] - pd.Timedelta(days=3),
                    row['date'] + pd.Timedelta(days=3),
                    alpha=0.4 * row['confidence'] + 0.1,
                    color=color, linewidth=0)

    ax1.plot(regime_df['date'], regime_df['signal_value'],
             'o-', color='black', markersize=5, linewidth=1, alpha=0.6)
    ax1.set_yticks([-1, 0, 1])
    ax1.set_yticklabels(['risk_off', 'neutral', 'risk_on'])
    ax1.set_ylabel('Regime')
    ax1.set_title('Macro Regime Classification — MacroAgent History', fontsize=12)
    ax1.grid(axis='x', alpha=0.3)

    patches = [mpatches.Patch(color=c, label=l) for v, c in REGIME_COLORS.items()
               for lv, l in REGIME_LABELS.items() if lv == v]
    ax1.legend(handles=patches, loc='upper right', fontsize=9)

    # ── Confidence (bottom) ────────────────────────────────────────────────
    ax2.fill_between(regime_df['date'], regime_df['confidence'],
                     alpha=0.5, color='steelblue')
    ax2.set_ylim(0, 1)
    ax2.set_ylabel('Confidence')
    ax2.set_xlabel('Date')
    ax2.grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Summary table
    regime_df['regime'] = regime_df['signal_value'].map(REGIME_LABELS)
    regime_df['rate_outlook'] = regime_df['rate_val'].map(RATE_MARKERS)
    display(
        regime_df[['date', 'regime', 'rate_outlook', 'confidence']]
        .assign(date=regime_df['date'].dt.strftime('%Y-%m-%d'))
        .rename(columns={'date': 'Date', 'regime': 'Regime',
                          'rate_outlook': 'Rate Outlook', 'confidence': 'Confidence'})
    )

## Section 5 — All signals in DB

In [ ]:
all_sig_df = pd.read_sql(
    text("""
        SELECT s.date, s.agent_name, s.target, s.signal_value, s.confidence,
               a.model_string, a.cached, a.cost_usd, a.latency_ms
        FROM signals s
        LEFT JOIN agent_calls a ON s.raw_call_id = a.call_id
        ORDER BY s.date DESC, s.agent_name, s.target
    """),
    engine,
)

if all_sig_df.empty:
    print('⚠  No signals in DB yet.')
else:
    all_sig_df['signal_value'] = all_sig_df['signal_value'].round(3)
    all_sig_df['cost_usd'] = all_sig_df['cost_usd'].map(lambda x: f'${x:.5f}' if pd.notna(x) else '—')
    all_sig_df['latency_ms'] = all_sig_df['latency_ms'].map(lambda x: f'{x:.0f}ms' if pd.notna(x) else '—')
    all_sig_df['cached'] = all_sig_df['cached'].map(lambda x: '✓' if x else '✗')
    display(all_sig_df)
    print(f'\nTotal signal rows: {len(all_sig_df)}')